# 66 — Hallucination Testing
**Goal:** Detect and quantify LLM hallucinations in resume output.

## 1. Types of Hallucination

In [ ]:
print('''Hallucination types in resume LLM tasks:
1. Skill hallucination — claiming a skill not in the resume
2. Experience hallucination — inventing job details
3. Date hallucination — wrong or made-up dates
4. Metric hallucination — fabricated numbers (40% improvement)

Detection strategies:
- Grounding check: Does output exist in source text?
- Consistency check: Same facts across multiple calls?
- Factual check: Known facts (e.g., company locations)''')

## 2. Grounding Checker

In [ ]:
import re

def grounding_check(output, source_text):
    """Check if output facts are grounded in source text."""
    # Tokenize into claims
    claims = re.findall(r"\\b[A-Z][a-z]+(?:\\s+[A-Z][a-z]+)*\\b", output)
    source_lower = source_text.lower()
    
    grounded = []
    ungrounded = []
    for claim in claims:
        if len(claim) < 3: continue
        if claim.lower() in source_lower:
            grounded.append(claim)
        else:
            ungrounded.append(claim)
    
    return {
        "grounded": grounded[:5],
        "ungrounded": ungrounded[:5],
        "grounded_pct": len(grounded) / max(len(grounded + ungrounded), 1),
    }

source = "Python developer with NLP experience at Google"
output = "Python developer with NLP and TensorFlow experience at Google (5 years)"
result = grounding_check(output, source)
print(f"Grounded: {result['grounded_pct']:.0%}")
print(f"Grounded facts: {result['grounded']}")
print(f"Ungrounded: {result['ungrounded']} (possible hallucinations)")

## 3. Consistency Testing

In [ ]:
def consistency_check(text, variations):
    """Check if same facts are preserved across variations."""
    # Extract key facts from original
    skills = re.findall(r"\\b(Python|Java|NLP|TensorFlow|AWS|Docker)\\b", text, re.IGNORECASE)
    skills = set(s.lower() for s in skills)
    
    consistent = 0
    total = 0
    for var in variations:
        var_skills = set(s.lower() for s in re.findall(r"\\b(Python|Java|NLP|TensorFlow|AWS|Docker)\\b", var, re.IGNORECASE))
        if skills == var_skills:
            consistent += 1
        total += 1
    
    return {
        "original_skills": list(skills),
        "consistent_runs": consistent,
        "total_runs": total,
        "consistency": consistent / max(total, 1),
    }

original = "Python and NLP expert with TensorFlow"
variations = [
    "Python NLP specialist, TensorFlow",
    "Expert in Python and TensorFlow with NLP focus",
    "Java developer with Spring Boot",  # Wrong!
]
result = consistency_check(original, variations)
print(f"Consistency: {result['consistency']:.0%} ({result['consistent_runs']}/{result['total_runs']})")
print(f"Variant 3 was inconsistent (Java vs Python/TensorFlow/NLP)")

## Summary: Hallucination testing catches LLM fabrications. Always ground-check outputs against source text.